In [94]:
#change projection

import arcpy
import os

# --- PATHS ---
gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
master_layer = os.path.join(gdb, "SEFM_events_94_24")
raw_permits = os.path.join(gdb, "ALLStates_BurnData") 

output_name = "Burn_Permits_AES_WGS84"
projected_permits = os.path.join(gdb, output_name)

# --- STEP 1: GET TARGET SPATIAL REFERENCE FROM MASTER LAYER (AES_WGS84) ---
print("Reading master spatial reference system...")
master_sr = arcpy.Describe(master_layer).spatialReference
print(f"Target Master Projection (SEFM): {master_sr.name}")

# --- STEP 2: DESCRIBE RAW PERMITS SPATIAL REFERENCE (Albers) ---
print(f"Describing incoming permit layer: {os.path.basename(raw_permits)}...")
raw_desc = arcpy.Describe(raw_permits)
raw_sr = raw_desc.spatialReference
print(f"Incoming Permits Projection: {raw_sr.name}")

# --- STEP 3: COMPARE AND PROJECT ---
if raw_sr.name == master_sr.name:
    print("-" * 60)
    print("MATCH CONFIRMED: The incoming permits are already using the correct AES_WGS84 projection!")
    if not arcpy.Exists(projected_permits):
        print(f"Copying layer into geodatabase as '{output_name}'...")
        arcpy.management.CopyFeatures(raw_permits, projected_permits)
else:
    print("-" * 60)
    print("PROJECTION MISMATCH DETECTED. Re-projecting permits to match master SEFM layer...")
    
    if arcpy.Exists(projected_permits):
        print(f"Target feature class '{output_name}' already exists. Skipping processing.")
    else:
        # Running the transformation to shift from Albers Conic -> AES WGS84
        arcpy.management.Project(
            in_dataset=raw_permits,
            out_dataset=projected_permits,
            out_coor_system=master_sr
        )
        print(f"SUCCESS: Projected permits dataset created at: {projected_permits}")

print("-" * 60)

Reading master spatial reference system...
Target Master Projection (SEFM): AEA_WGS84
Describing incoming permit layer: ALLStates_BurnData...
Incoming Permits Projection: USA_Contiguous_Albers_Equal_Area_Conic
------------------------------------------------------------
PROJECTION MISMATCH DETECTED. Re-projecting permits to match master SEFM layer...
SUCCESS: Projected permits dataset created at: C:\GIS\ClassiFIRE\Project\ClassiFIRE.gdb\Burn_Permits_AES_WGS84
------------------------------------------------------------


In [95]:
# clip to extent of SE Firemap

import arcpy
import os

# --- PATHS ---
# clip to extent of SE Firemap

import arcpy
import os

# --- PATHS ---
gdb               = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
projected_permits = os.path.join(gdb, "Burn_Permits_AES_WGS84")
clip_features     = os.path.join(gdb, "extent_Dissolved")

output_name       = "Burn_Permits_Clipped"
clipped_output    = os.path.join(gdb, output_name)

# --- STEP 1: VERIFY INPUTS EXIST ---
print("Verifying clipping footprints...")
if not arcpy.Exists(projected_permits):
    raise FileNotFoundError(f"Missing input layer: {projected_permits}")
if not arcpy.Exists(clip_features):
    raise FileNotFoundError(f"Missing clipping extent layer: {clip_features}")

# --- STEP 2: EXECUTE VECTOR CLIP ---
if arcpy.Exists(clipped_output):
    print(f"Target feature class '{output_name}' already exists. Skipping clipping process.")
else:
    print(f"Clipping permits layer to the extent of {os.path.basename(clip_features)}...")
    
    arcpy.analysis.Clip(
        in_features=projected_permits,
        clip_features=clip_features,
        out_feature_class=clipped_output
    )
    
    # Quick row count check to see how much data we trimmed
    raw_count = int(arcpy.management.GetCount(projected_permits)[0])
    clipped_count = int(arcpy.management.GetCount(clipped_output)[0])
    trimmed = raw_count - clipped_count
    
    print("-" * 60)
    print(f"SUCCESS: Clipped permits dataset created at: {clipped_output}")
    print(f"  -> Original Permit Records: {raw_count:,}")
    print(f"  -> Remaining Study Area Records: {clipped_count:,}")
    print(f"  -> Extraneous Records Trimmed: {trimmed:,}")
    print("-" * 60)

# Flush any schema links
arcpy.management.ClearWorkspaceCache(gdb)
projected_permits = os.path.join(gdb, "Burn_Permits_AES_WGS84")
clip_features     = os.path.join(gdb, "extent_Dissolved")

output_name       = "Burn_Permits_Clipped"
clipped_output    = os.path.join(gdb, output_name)

# --- STEP 1: VERIFY INPUTS EXIST ---
print("Verifying clipping footprints...")
if not arcpy.Exists(projected_permits):
    raise FileNotFoundError(f"Missing input layer: {projected_permits}")
if not arcpy.Exists(clip_features):
    raise FileNotFoundError(f"Missing clipping extent layer: {clip_features}")

# --- STEP 2: EXECUTE VECTOR CLIP ---
if arcpy.Exists(clipped_output):
    print(f"Target feature class '{output_name}' already exists. Skipping clipping process.")
else:
    print(f"Clipping permits layer to the extent of {os.path.basename(clip_features)}...")
    
    arcpy.analysis.Clip(
        in_features=projected_permits,
        clip_features=clip_features,
        out_feature_class=clipped_output
    )
    
    # Quick row count check to see how much data we trimmed
    raw_count = int(arcpy.management.GetCount(projected_permits)[0])
    clipped_count = int(arcpy.management.GetCount(clipped_output)[0])
    trimmed = raw_count - clipped_count
    
    print("-" * 60)
    print(f"SUCCESS: Clipped permits dataset created at: {clipped_output}")
    print(f"  -> Original Permit Records: {raw_count:,}")
    print(f"  -> Remaining Study Area Records: {clipped_count:,}")
    print(f"  -> Extraneous Records Trimmed: {trimmed:,}")
    print("-" * 60)

# Flush any schema links
arcpy.management.ClearWorkspaceCache(gdb)

Verifying clipping footprints...
Clipping permits layer to the extent of extent_Dissolved...
------------------------------------------------------------
SUCCESS: Clipped permits dataset created at: C:\GIS\ClassiFIRE\Project\ClassiFIRE.gdb\Burn_Permits_Clipped
  -> Original Permit Records: 2,600,607
  -> Remaining Study Area Records: 2,582,869
  -> Extraneous Records Trimmed: 17,738
------------------------------------------------------------


<Result 'true'>

In [96]:
#how many records have acres = NULL?
import arcpy
import os

gdb     = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
permits = os.path.join(gdb, "Burn_Permits_Clipped")
field   = "ACRES"

print(f"Auditing '{field}' column for missing data...")

# Verify the field name exists and get its type to ensure accurate querying
fields_list = [f.name for f in arcpy.ListFields(permits)]
if field not in fields_list:
    raise AttributeError(f"Could not find field '{field}' in the permit layer. Available fields: {fields_list}")

null_count = 0
zero_count = 0
valid_count = 0
total_records = 0

with arcpy.da.SearchCursor(permits, [field]) as cur:
    for row in cur:
        total_records += 1
        val = row[0]
        
        if val is None:
            null_count += 1
        elif val == 0:
            zero_count += 1
        else:
            valid_count += 1

# Calculate percentages for context
null_pct = (null_count / total_records) * 100 if total_records > 0 else 0
zero_pct = (zero_count / total_records) * 100 if total_records > 0 else 0

print("-" * 60)
print(f"Total Clipped Permit Records: {total_records:,}")
print(f"Strict Database NULLs:        {null_count:,} ({null_pct:.2f}%)")
print(f"Numeric Zeros (0.0):          {zero_count:,} ({zero_pct:.2f}%)")
print(f"Valid Populated Acreages:    {valid_count:,} ({(valid_count/total_records)*100:.2f}%)")
print("-" * 60)

Auditing 'ACRES' column for missing data...
------------------------------------------------------------
Total Clipped Permit Records: 2,582,869
Strict Database NULLs:        706,504 (27.35%)
Numeric Zeros (0.0):          201,369 (7.80%)
Valid Populated Acreages:    1,674,996 (64.85%)
------------------------------------------------------------


In [97]:
#filter out acres = NULL

import arcpy
import os

# --- PATHS ---
gdb          = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
in_permits   = os.path.join(gdb, "Burn_Permits_Clipped")
output_name  = "Burn_Permits_Filtered"
out_permits  = os.path.join(gdb, output_name)

field = "ACRES"

# --- STEP 1: CHECK IF OUTPUT ALREADY EXISTS ---
if arcpy.Exists(out_permits):
    print(f"Target feature class '{output_name}' already exists. Skipping filtering process.")
else:
    print(f"Isolating records where {field} is NOT NULL...")
    
    # Create an in-memory feature layer to perform the selection
    temp_layer = "permits_temp_layer"
    arcpy.management.MakeFeatureLayer(in_permits, temp_layer)
    
    # Select rows where ACRES is not null
    sql_query = f"{field} IS NOT NULL"
    arcpy.management.SelectLayerByAttribute(temp_layer, "NEW_SELECTION", sql_query)
    
    # Save the selected features to the new feature class
    print(f"Exporting clean records to '{output_name}'...")
    arcpy.management.CopyFeatures(temp_layer, out_permits)
    
    # Clean up the temporary in-memory layer
    arcpy.management.Delete(temp_layer)
    
    # Verify row counts
    original_count = int(arcpy.management.GetCount(in_permits)[0])
    filtered_count = int(arcpy.management.GetCount(out_permits)[0])
    removed_count = original_count - filtered_count
    
    print("-" * 60)
    print(f"SUCCESS: Filtered permits dataset created at: {out_permits}")
    print(f"  -> Input Permit Records: {original_count:,}")
    print(f"  -> Clean Records Retained: {filtered_count:,}")
    print(f"  -> NULL Acreage Records Removed: {removed_count:,}")
    print("-" * 60)

# Clear database locks
arcpy.management.ClearWorkspaceCache(gdb)

Isolating records where ACRES is NOT NULL...
Exporting clean records to 'Burn_Permits_Filtered'...
------------------------------------------------------------
SUCCESS: Filtered permits dataset created at: C:\GIS\ClassiFIRE\Project\ClassiFIRE.gdb\Burn_Permits_Filtered
  -> Input Permit Records: 2,582,869
  -> Clean Records Retained: 1,876,365
  -> NULL Acreage Records Removed: 706,504
------------------------------------------------------------


<Result 'true'>

In [99]:
# calculate area in ha. count how many are smaller than 0.809 (2 acres) and then exclude them. 

import arcpy
import os

# --- PATHS ---
gdb            = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
in_features    = os.path.join(gdb, "Burn_Permits_Filtered")
output_name    = "burn_permits_large"
out_features   = os.path.join(gdb, output_name)

acres_field    = "ACRES"
target_field   = "area_ha"
threshold_ha   = 0.809
conversion_factor = 0.404686

# --- STEP 1: ADD FIELD IF MISSING ---
existing_fields = [f.name for f in arcpy.ListFields(in_features)]
if target_field not in existing_fields:
    print(f"Adding DOUBLE field '{target_field}'...")
    arcpy.management.AddField(in_features, target_field, "DOUBLE", field_alias="area_ha")
else:
    print(f"Field '{target_field}' already exists. Proceeding to calculation...")

# --- STEP 2: CALCULATE HECTARES FROM ACRES ---
print(f"Calculating hectares from '{acres_field}' column via math cursor...")
with arcpy.da.UpdateCursor(in_features, [acres_field, target_field]) as calc_cur:
    for row in calc_cur:
        acres_val = row[0]
        if acres_val is not None:
            # Direct math conversion for point attributes
            row[1] = float(acres_val) * conversion_factor
            calc_cur.updateRow(row)

print("Hectares calculation complete.")

# --- STEP 3: AUDIT AND EXCLUDE SMALL PERMIT POINTS ---
if arcpy.Exists(out_features):
    print(f"Target feature class '{output_name}' already exists. Skipping final export.")
else:
    print(f"Evaluating point records smaller than {threshold_ha} ha...")
    
    # Setup temporary layer to run attribute selection
    temp_layer = "permits_ha_layer"
    arcpy.management.MakeFeatureLayer(in_features, temp_layer)
    
    # Select features that meet or exceed our broadcast-size cutoff
    sql_query = f"{target_field} >= {threshold_ha}"
    arcpy.management.SelectLayerByAttribute(temp_layer, "NEW_SELECTION", sql_query)
    
    print(f"Exporting broadcast permit points (>= {threshold_ha} ha) to '{output_name}'...")
    arcpy.management.CopyFeatures(temp_layer, out_features)
    
    # Clean up the temporary in-memory layer
    arcpy.management.Delete(temp_layer)
    
    # Calculate operational metrics
    total_before = int(arcpy.management.GetCount(in_features)[0])
    total_after  = int(arcpy.management.GetCount(out_features)[0])
    excluded     = total_before - total_after
    
    print("-" * 60)
    print(f"SUCCESS: Final broadcast permit points layer created.")
    print(f"  -> Total Inputs Processed:  {total_before:,}")
    print(f"  -> Broadcast Permits Saved: {total_after:,}")
    print(f"  -> Small Permits Excluded:  {excluded:,} ({(excluded/total_before)*100:.2f}%)")
    print("-" * 60)

# Clear database memory locks
arcpy.management.ClearWorkspaceCache(gdb)

Field 'area_ha' already exists. Proceeding to calculation...
Calculating hectares from 'ACRES' column via math cursor...
Hectares calculation complete.
Evaluating point records smaller than 0.809 ha...
Exporting broadcast permit points (>= 0.809 ha) to 'burn_permits_large'...
------------------------------------------------------------
SUCCESS: Final broadcast permit points layer created!
  -> Total Inputs Processed:  1,876,365
  -> Broadcast Permits Saved: 1,150,597
  -> Small Permits Excluded:  725,768 (38.68%)
------------------------------------------------------------


<Result 'true'>

In [100]:
# ===================================================================
# Count how many records in burn_permits_large have 
#          NULL values in the DATE column.
# ===================================================================
import arcpy
import os

gdb     = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
permits = os.path.join(gdb, "burn_permits_large")
field   = "DATE"

print(f"Auditing '{field}' column for missing timestamps...")

# Verify the field name exists
fields_list = [f.name for f in arcpy.ListFields(permits)]
if field not in fields_list:
    raise AttributeError(f"Could not find field '{field}' in the permit layer. Available fields: {fields_list}")

null_count = 0
valid_count = 0
total_records = 0

with arcpy.da.SearchCursor(permits, [field]) as cur:
    for row in cur:
        total_records += 1
        val = row[0]
        
        if val is None:
            null_count += 1
        else:
            valid_count += 1

# Calculate percentages for context
null_pct = (null_count / total_records) * 100 if total_records > 0 else 0
valid_pct = (valid_count / total_records) * 100 if total_records > 0 else 0

print("-" * 60)
print(f"Total Large Permit Records:   {total_records:,}")
print(f"Missing Dates (NULL):         {null_count:,} ({null_pct:.2f}%)")
print(f"Valid Populated Dates:        {valid_count:,} ({valid_pct:.2f}%)")
print("-" * 60)

Auditing 'DATE' column for missing timestamps...
------------------------------------------------------------
Total Large Permit Records:   1,150,597
Missing Dates (NULL):         2,800 (0.24%)
Valid Populated Dates:        1,147,797 (99.76%)
------------------------------------------------------------


In [101]:
# ===================================================================
# create start_date. It's DATE if not NULL. If DATE is NULL, it's Jan 1 of YEAR.
import arcpy
from datetime import datetime
import os

gdb          = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
permits      = os.path.join(gdb, "burn_permits_large")

target_field = "start_date"

# --- STEP 1: ADD START_DATE FIELD IF MISSING ---
existing_fields = [f.name for f in arcpy.ListFields(permits)]
if target_field not in existing_fields:
    print(f"Adding DATE field '{target_field}'...")
    arcpy.management.AddField(permits, target_field, "DATE", field_alias="start_date")
else:
    print(f"Field '{target_field}' already exists. Overwriting with calculation logic...")

# --- STEP 2: CONVERT AND POPULATE LOGIC ---
print("Processing date conditional logic...")
mapped_from_date = 0
fallback_to_year = 0
skipped_corrupt  = 0

fields = ["DATE", "YEAR", target_field]

with arcpy.da.UpdateCursor(permits, fields) as cur:
    for row in cur:
        date_val = row[0]
        year_val = row[1]
        
        # Scenario A: Valid timestamp exists in DATE field
        if date_val is not None:
            row[2] = date_val
            mapped_from_date += 1
            
        # Scenario B: DATE is NULL, fall back to Jan 1 of the YEAR column
        elif year_val is not None and str(year_val).strip() != "":
            try:
                # Clean up year formatting (handles floats like 2015.0 or strings safely)
                clean_year = int(float(str(year_val).strip()))
                # Construct datetime object for January 1st
                row[2] = datetime(clean_year, 1, 1)
                fallback_to_year += 1
            except ValueError:
                # Handle cases where the year string might be non-numeric or corrupt
                row[2] = None
                skipped_corrupt += 1
                
        # Scenario C: Both fields are completely missing
        else:
            row[2] = None
            skipped_corrupt += 1
            
        cur.updateRow(row)

print("-" * 60)
print(f"SUCCESS: '{target_field}' field population complete.")
print(f"  -> Direct mappings from 'DATE':       {mapped_from_date:,}")
print(f"  -> Fallback mappings to Jan 1 'YEAR': {fallback_to_year:,}")
if skipped_corrupt > 0:
    print(f"  -> Records skipped (no valid data):   {skipped_corrupt:,}")
print("-" * 60)

# Clear database locks
arcpy.management.ClearWorkspaceCache(gdb)

Adding DATE field 'start_date'...
Processing date conditional logic...
------------------------------------------------------------
SUCCESS: 'start_date' field population complete!
  -> Direct mappings from 'DATE':       1,147,797
  -> Fallback mappings to Jan 1 'YEAR': 2,800
------------------------------------------------------------


<Result 'true'>

In [102]:
# ===================================================================
# Create end_date column and populate it based on DATE,
#          falling back to Dec 31 of YEAR if DATE is NULL. 
# ===================================================================
import arcpy
from datetime import datetime
import os

gdb          = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
permits      = os.path.join(gdb, "burn_permits_large")

target_field = "end_date"

# --- STEP 1: ADD END_DATE FIELD (NO ALIAS) ---
existing_fields = [f.name for f in arcpy.ListFields(permits)]
if target_field not in existing_fields:
    print(f"Adding DATE field '{target_field}'...")
    # Passing the exact field name to field_alias prevents descriptive labels
    arcpy.management.AddField(permits, target_field, "DATE", field_alias=target_field)
else:
    print(f"Field '{target_field}' already exists. Overwriting with calculation logic...")

# --- STEP 2: CONVERT AND POPULATE LOGIC ---
print("Processing end date conditional logic...")
mapped_from_date = 0
fallback_to_year = 0
skipped_corrupt  = 0

fields = ["DATE", "YEAR", target_field]

with arcpy.da.UpdateCursor(permits, fields) as cur:
    for row in cur:
        date_val = row[0]
        year_val = row[1]
        
        # Scenario A: Valid timestamp exists in DATE field
        if date_val is not None:
            row[2] = date_val
            mapped_from_date += 1
            
        # Scenario B: DATE is NULL, fall back to Dec 31 of the YEAR column
        elif year_val is not None and str(year_val).strip() != "":
            try:
                clean_year = int(float(str(year_val).strip()))
                # Construct datetime object for December 31st
                row[2] = datetime(clean_year, 12, 31)
                fallback_to_year += 1
            except ValueError:
                row[2] = None
                skipped_corrupt += 1
                
        # Scenario C: Both fields are missing
        else:
            row[2] = None
            skipped_corrupt += 1
            
        cur.updateRow(row)

print("-" * 60)
print(f"SUCCESS: '{target_field}' field population complete.")
print(f"  -> Direct mappings from 'DATE':       {mapped_from_date:,}")
print(f"  -> Fallback mappings to Dec 31 'YEAR': {fallback_to_year:,}")
if skipped_corrupt > 0:
    print(f"  -> Records skipped (no valid data):   {skipped_corrupt:,}")
print("-" * 60)

# Clear database locks
arcpy.management.ClearWorkspaceCache(gdb)

Adding DATE field 'end_date'...
Processing end date conditional logic...
------------------------------------------------------------
SUCCESS: 'end_date' field population complete!
  -> Direct mappings from 'DATE':       1,147,797
  -> Fallback mappings to Dec 31 'YEAR': 2,800
------------------------------------------------------------


<Result 'true'>

In [103]:
# ===================================================================
# Scan the entire dataset to identify and count stacked 
#          coordinates representing artificial county/zip centroids.
# ===================================================================
import arcpy
import os
from collections import Counter

gdb     = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
permits = os.path.join(gdb, "burn_permits_large")

# Using the native geometry token to extract raw coordinate pairs
fields = ["SHAPE@XY", "YEAR"]

print("Initiating full-dataset spatial distribution scan...")


coord_counter = Counter()
total_records = 0

with arcpy.da.SearchCursor(permits, fields) as cur:
    for row in cur:
        total_records += 1
        coord = row[0]
        if coord is not None:
            # Rounding coordinates slightly ensures we catch points that vary by fractions of a millimeter
            # due to database rounding, while still treating them as the same physical spot.
            rounded_coord = (round(coord[0], 3), round(coord[1], 3))
            coord_counter[rounded_coord] += 1

# Defining a centroid cluster as any single coordinate point containing more than 50 permits
centroid_threshold = 50
stacked_points = {k: v for k, v in coord_counter.items() if v >= centroid_threshold}
total_trapped_records = sum(stacked_points.values())

print("-" * 60)
print(f"Total Dataset Records Evaluated: {total_records:,}")
print(f"Unique Artificial Centroid Spots: {len(stacked_points):,}")
print(f"Total Permits Trapped on Centroids: {total_trapped_records:,} ({(total_trapped_records/total_records)*100:.2f}%)")
print("-" * 60)

# Optional breakdown: Show top 10 worst offending coordinates
print("Top 10 Largest Point Stacks Found:")
for i, (coord, count) in enumerate(coord_counter.most_common(10), 1):
    print(f"  {i}. Coordinates: {coord} -> {count:,} permits stacked")
print("-" * 60)

Initiating full-dataset spatial distribution scan...
This may take a moment to read all records into memory...
------------------------------------------------------------
Total Dataset Records Evaluated: 1,150,597
Unique Artificial Centroid Spots: 301
Total Permits Trapped on Centroids: 361,084 (31.38%)
------------------------------------------------------------
Top 10 Largest Point Stacks Found:
  1. Coordinates: (1149064.893, 938444.559) -> 13,876 permits stacked
  2. Coordinates: (1086431.836, 932307.224) -> 11,726 permits stacked
  3. Coordinates: (1119107.934, 935898.852) -> 11,116 permits stacked
  4. Coordinates: (1118078.913, 975076.918) -> 9,545 permits stacked
  5. Coordinates: (1158676.348, 976089.095) -> 9,036 permits stacked
  6. Coordinates: (1145780.343, 1015169.36) -> 8,941 permits stacked
  7. Coordinates: (1288962.002, 1057000.642) -> 7,816 permits stacked
  8. Coordinates: (1239706.239, 1027487.824) -> 7,149 permits stacked
  9. Coordinates: (1181452.396, 940209.84

In [108]:
# select stacked coordinates so I can look at them. 

# ===================================================================
# SCRIPT STEP: LF_SEFM_19f_Select_Centroids_Final_Fix.py
# PURPOSE: Select centroid-trapped records visually on the map using
#          the strictly lowercase arcpy.management tool naming.
# ===================================================================
import arcpy
import os
from collections import Counter

gdb          = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
layer_name   = "burn_permits_large" # The layer active in map 
permits_path = os.path.join(gdb, layer_name)

print("Scanning for stacked coordinates to map out centroids...")
coord_counter = Counter()

# Step 1: Track down the centroid coordinate points
spatial_ref = arcpy.Describe(permits_path).spatialReference
with arcpy.da.SearchCursor(permits_path, ["SHAPE@XY"]) as cur:
    for row in cur:
        coord = row[0]
        if coord is not None:
            rounded_coord = (round(coord[0], 3), round(coord[1], 3))
            coord_counter[rounded_coord] += 1

blacklist = {coord for coord, count in coord_counter.items() if count >= 50}
print(f"Found {len(blacklist)} centroid points. Creating temporary selection targets...")

# Step 2: Build a temporary, in-memory point layer of just the 301 centroids
temp_centroids = r"memory\centroid_targets"
if arcpy.Exists(temp_centroids):
    arcpy.management.Delete(temp_centroids)

# FIX: Notice the lowercase 'c' in CreateFeatureclass
arcpy.management.CreateFeatureclass(
    out_path="memory", 
    out_name="centroid_targets", 
    geometry_type="POINT", 
    spatial_reference=spatial_ref
)

with arcpy.da.InsertCursor(temp_centroids, ["SHAPE@XY"]) as i_cur:
    for coord in blacklist:
        i_cur.insertRow([coord])

# Step 3: Run Select Layer By Location 
print(f"Selecting stacked points in '{layer_name}' via spatial intersection...")
try:
    # Selects any point in your permit layer that sits exactly on our centroid targets
    arcpy.management.SelectLayerByLocation(
        in_layer=layer_name,
        overlap_type="ARE_IDENTICAL_TO",
        select_features=temp_centroids,
        search_distance=None,
        selection_type="NEW_SELECTION"
    )
    
    print("-" * 60)
    print(f"SUCCESS: Centroid records selected.")
    print(" -> Flip over to  ArcGIS Pro map window.")
    print(" -> The 361,084 centroid-trapped points will now be highlighted.")
    print("-" * 60)

except Exception as e:
    print(f"\nSpatial selection failed: {e}")
    print("Ensure 'burn_permits_large' is explicitly dragged into your active Map Contents pane.")

finally:
    # Clean up the memory workspace
    if arcpy.Exists(temp_centroids):
        arcpy.management.Delete(temp_centroids)

Scanning for stacked coordinates to map out centroids...
Found 301 centroid points. Creating temporary selection targets...
Selecting stacked points in 'burn_permits_large' via spatial intersection...
------------------------------------------------------------
SUCCESS: Centroid records selected flawlessly!
 -> Flip over to your ArcGIS Pro map window.
 -> Your 361,084 centroid-trapped points will now be highlighted.
------------------------------------------------------------


In [109]:
#exclude permits where 50 or more are stacked in the same location. Most are in GA. 

import arcpy
import os

gdb          = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
layer_name   = "burn_permits_large" # Active map layer with centroids selected
output_name  = "burn_permits_stacked_removed"
out_features = os.path.join(gdb, output_name)

if arcpy.Exists(out_features):
    print(f"Target feature class '{output_name}' already exists. Skipping export.")
else:
    try:
        print("Inverting selection to target true geographic locations...")
        # SWITCH_SELECTION drops the 361k centroids and selects the remaining 789k clean points
        arcpy.management.SelectLayerByAttribute(layer_name, "SWITCH_SELECTION")
        
        # Verify selection count matches expectations before running the write operation
        selected_count = int(arcpy.management.GetCount(layer_name)[0])
        print(f"Verified selection: {selected_count:,} clean records targeted.")
        
        print(f"Exporting clean records to '{output_name}'...")
        # CopyFeatures respects active selections and only exports what is highlighted
        arcpy.management.CopyFeatures(layer_name, out_features)
        
        # Clear the selection on the original layer to tidy up the map workspace
        arcpy.management.SelectLayerByAttribute(layer_name, "CLEAR_SELECTION")
        
        print("-" * 60)
        print(f"SUCCESS: Clean burn permits layer created.")
        print(f"  -> Destination:      {out_features}")
        print(f"  -> Validated Rows:   {selected_count:,}")
        print("-" * 60)
        
    except Exception as e:
        print(f"\nExport failed: {e}")
        print("Make sure 'burn_permits_large' is still active in map pane with its selection intact.")

# Clear database memory locks
arcpy.management.ClearWorkspaceCache(gdb)

Inverting selection to target true geographic locations...
Verified selection: 789,490 clean records targeted.
Exporting clean records to 'burn_permits_stacked_removed'...
------------------------------------------------------------
SUCCESS: Clean burn permits layer created!
  -> Destination:      C:\GIS\ClassiFIRE\Project\ClassiFIRE.gdb\burn_permits_stacked_removed
  -> Validated Rows:   789,490
------------------------------------------------------------


<Result 'true'>

In [1]:
# ===================================================================
# add unique permit id
import os
import arcpy

gdb     = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
permits = os.path.join(gdb, "burn_permits_stacked_removed")

target_field = "permit_id"

# --- STEP 1: ADD FIELD ---
existing_fields = [f.name for f in arcpy.ListFields(permits)]
if target_field not in existing_fields:
    print(f"Adding permanent long-integer field '{target_field}'...")
    arcpy.management.AddField(permits, target_field, "LONG")
else:
    print(f"Field '{target_field}' already exists. Recalculating IDs...")

# --- STEP 2: POPULATE SEQUENTIAL IDS ---
print("Populating unique IDs across all records...")
id_counter = 1

# Using an UpdateCursor ensures fast, sequential processing row-by-row
with arcpy.da.UpdateCursor(permits, [target_field]) as cur:
    for row in cur:
        row[0] = id_counter
        cur.updateRow(row)
        id_counter += 1

print("-" * 60)
print(f"SUCCESS: '{target_field}' populated! {id_counter - 1:,} records indexed.")
print("-" * 60)

# Flush cache to commit changes immediately
arcpy.management.ClearWorkspaceCache(gdb)

Adding permanent long-integer field 'permit_id'...
Populating unique IDs across all records...
------------------------------------------------------------
SUCCESS: 'permit_id' populated! 789,490 records indexed.
------------------------------------------------------------


<Result 'true'>